# 🧬 ARC (Adaptive Retention & Correction) for Molecular Multi-Label Classification
### ICLR 2025 — Research-Grade Implementation with MolFormer

**Pipeline:**  
`Raw CSV → SMILES Cleaning → Preprocessing → Task Split → MolFormer Sequential Training → ARC Test-Time Adaptation → Evaluation + Ablation + Visualization`

**Datasets:** ToxCast (99 labels, MAIN) | Tox21 (12 labels) | SIDER (26 labels)  
**Model:** MolFormer (HuggingFace pretrained) → Linear Classifier (BCEWithLogitsLoss)  
**ARC:** Test-time only — Adaptive Retention + Adaptive Correction — No memory replay


In [1]:
import sys
print(sys.executable)

/home/student/miniforge3/envs/molformer/bin/python


In [2]:
import torch

print("======== GPU CHECK ========")

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current GPU index:", torch.cuda.current_device())
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version (torch):", torch.version.cuda)
else:
    print("❌ GPU not detected. Using CPU")

print("===========================")

======== GPU CHECK ========
CUDA available: True
GPU count: 1
Current GPU index: 0
GPU name: Tesla V100S-PCIE-32GB
CUDA version (torch): 12.1


In [3]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================
# Install required packages
# Run this cell once. Restart kernel after installation if needed.
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'scikit-learn>=1.3.0',
    'numpy>=1.24.0',
    'matplotlib>=3.7.0',
    'pandas>=2.0.0',
    'rdkit',
    'einops',                   # required by MolFormer
    'rotary-embedding-torch', 
    'requests' 
]

for pkg in packages:
    try:
        install(pkg)
        print(f'  ✓ {pkg}')
    except Exception as e:
        print(f'  ✗ {pkg}: {e}')

print('\nInstallation complete. Restart kernel if this is the first run.')


  ✓ torch>=2.0.0


  ✓ transformers>=4.35.0


  ✓ scikit-learn>=1.3.0


  ✓ numpy>=1.24.0


  ✓ matplotlib>=3.7.0


  ✓ pandas>=2.0.0


  ✓ rdkit


  ✓ einops


  ✓ rotary-embedding-torch


  ✓ requests

Installation complete. Restart kernel if this is the first run.


In [1]:
# ============================================================
# CELL 2 — GLOBAL IMPORTS
# ============================================================
import os, warnings, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from rdkit import Chem

warnings.filterwarnings("ignore")

# ── Device ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports done | Device: {DEVICE}")


✅ Imports done | Device: cuda


In [ ]:
# ============================================================
# CELL 3 — GLOBAL CONFIG
# ============================================================

# ── File paths (UPDATE THESE) ──
CONFIG = {
    "tox21_path":   "tox21.csv",
    "sider_path":   "sider.csv",
    "toxcast_path": "toxcast_data.csv",

    # MolFormer HuggingFace model
    "molformer_name": "ibm/MolFormer-XL-both-10pct",

    # Training
    "batch_size":   32,
    "lr_backbone":  1e-5,
    "lr_head":      1e-3,
    "epochs":       20,          # increase for real experiments
    "max_length":   128,
    "seed":         42,

    # ARC thresholds (from paper ablation, robust in 0.6–0.9 range)
    "epsilon":  0.8,   # Assumption 1 — Retention threshold
    "theta":    0.7,   # Assumption 2 — Correction threshold
    "tss_temp": 2.0,    # TSS temperature > 1

    # Label filtering
    "nan_threshold":     0.5,   # drop cols with >50% NaN
    "density_threshold": 0.03,  # keep labels with >3% positive rate (ToxCast)
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
print("✅ Config set")
print(json.dumps({k: v for k, v in CONFIG.items() if k not in ["tox21_path","sider_path","toxcast_path"]}, indent=2))


✅ Config set
{
  "molformer_name": "ibm/MolFormer-XL-both-10pct",
  "batch_size": 32,
  "lr_backbone": 1e-05,
  "lr_head": 0.001,
  "epochs": 20,
  "max_length": 128,
  "seed": 42,
  "epsilon": 0.85,
  "theta": 0.65,
  "tss_temp": 2.0,
  "nan_threshold": 0.5,
  "density_threshold": 0.03
}


## 📥 Section 1: Data Loading & Preprocessing
*(Exact pipeline from your `datapreprocessing.ipynb`)*

In [4]:
# ============================================================
# CELL 4 — LOAD CSV FILES
# ============================================================
def load_csv(path, name):
    print(f"\n📥 Loading {name}...")
    df = pd.read_csv(path)
    print(f"✅ {name} loaded | Shape: {df.shape}")
    return df

tox21_df   = load_csv(CONFIG["tox21_path"],   "Tox21")
sider_df   = load_csv(CONFIG["sider_path"],   "SIDER")
toxcast_df = load_csv(CONFIG["toxcast_path"], "ToxCast")



📥 Loading Tox21...
✅ Tox21 loaded | Shape: (7831, 15)

📥 Loading SIDER...
✅ SIDER loaded | Shape: (1427, 28)

📥 Loading ToxCast...
✅ ToxCast loaded | Shape: (8576, 618)


In [5]:
# ============================================================
# CELL 5 — SMILES CLEANING (RDKit canonical + invalid removal)
# ============================================================
def clean_smiles_column(df, smiles_col="SMILES"):
    """Remove invalid SMILES and convert to canonical form."""
    cleaned, valid_idx = [], []
    for idx, smi in tqdm(enumerate(df[smiles_col]), total=len(df)):
        if pd.isna(smi) or smi == "":
            continue
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            continue
        cleaned.append(Chem.MolToSmiles(mol))
        valid_idx.append(idx)

    df_clean = df.iloc[valid_idx].copy()
    df_clean[smiles_col] = cleaned
    df_clean.reset_index(drop=True, inplace=True)
    print(f"Original: {len(df)} | Valid: {len(df_clean)} | Removed: {len(df)-len(df_clean)}")
    return df_clean

tox21_clean   = clean_smiles_column(tox21_df)
toxcast_clean = clean_smiles_column(toxcast_df)
sider_clean   = clean_smiles_column(sider_df)

# Drop duplicates
tox21_clean   = tox21_clean.drop_duplicates(subset="SMILES")
toxcast_clean = toxcast_clean.drop_duplicates(subset="SMILES")
sider_clean   = sider_clean.drop_duplicates(subset="SMILES")

print("\n✅ Final sizes after dedup:")
print(f"  Tox21: {len(tox21_clean)}  | ToxCast: {len(toxcast_clean)}  | SIDER: {len(sider_clean)}")


 26%|██▌       | 2015/7831 [00:00<00:01, 5097.37it/s][17:00:29] Explicit valence for atom # 3 Al, 6, is greater than permitted
[17:00:29] Explicit valence for atom # 4 Al, 6, is greater than permitted
 58%|█████▊    | 4526/7831 [00:00<00:00, 4965.80it/s][17:00:29] Explicit valence for atom # 9 Al, 6, is greater than permitted
[17:00:29] Explicit valence for atom # 5 Al, 6, is greater than permitted
100%|██████████| 7831/7831 [00:01<00:00, 4866.92it/s]


Original: 7831 | Valid: 7823 | Removed: 8


100%|██████████| 8576/8576 [00:01<00:00, 4926.38it/s]


Original: 8576 | Valid: 8576 | Removed: 0


  0%|          | 0/1427 [00:00<?, ?it/s][17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
 38%|███▊      | 548/1427 [00:00<00:00, 2751.55it/s][17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
[17:00:32] WARNING: not removing hydrogen atom without neighbors
 86%|████████▌ | 1221/1427 [00:00<00:00, 3166.23it/s][17:00:33] WARNING: not removing hydrogen atom without neighbors
[17:00:33] WARNING: not removing hydrogen atom without neighbors
100%|█████

Original: 1427 | Valid: 1427 | Removed: 0

✅ Final sizes after dedup:
  Tox21: 7823  | ToxCast: 8576  | SIDER: 1427


In [6]:
# ============================================================
# CELL 6 — PREPROCESSING: NaN handling, column filtering
# ============================================================
def preprocess_dataset(df, name, nan_threshold=0.5):
    """
    1. Replace -1 → NaN
    2. Drop high-NaN label columns (> nan_threshold)
    3. Drop rows with ALL labels NaN
    4. Fill remaining NaN with 0
    """
    print(f"\n⚙️  Preprocessing {name}...")
    df = df.copy()
    df.replace(-1, np.nan, inplace=True)

    non_label = ["SMILES", "mol_id", "Label"]
    label_cols = [c for c in df.columns if c not in non_label]

    nan_ratio = df[label_cols].isna().mean()
    keep_cols = nan_ratio[nan_ratio < nan_threshold].index.tolist()
    df = df[["SMILES"] + keep_cols].copy()
    print(f"  ✅ Kept {len(keep_cols)} labels after NaN filtering")

    df.dropna(subset=keep_cols, how="all", inplace=True)
    df[keep_cols] = df[keep_cols].fillna(0)
    df.reset_index(drop=True, inplace=True)

    print(f"  Final shape: {df.shape}")
    print(f"  Unique label values: {np.unique(df[keep_cols].values)}")
    return df, keep_cols

def filter_labels_by_density(df, label_cols, threshold=0.03):
    """Keep only labels with positive rate > threshold (ToxCast noise reduction)."""
    density = df[label_cols].sum() / len(df)
    keep = density[density > threshold].index.tolist()
    print(f"  Density filter: kept {len(keep)}/{len(label_cols)} labels")
    return keep

tox21_clean, tox21_labels     = preprocess_dataset(tox21_clean, "Tox21",   nan_threshold=CONFIG["nan_threshold"])
sider_clean, sider_labels     = preprocess_dataset(sider_clean, "SIDER",   nan_threshold=CONFIG["nan_threshold"])
toxcast_clean, toxcast_labels = preprocess_dataset(toxcast_clean, "ToxCast", nan_threshold=CONFIG["nan_threshold"])

# ToxCast extra density filter → 99 labels
toxcast_labels = filter_labels_by_density(
    toxcast_clean, toxcast_labels,
    threshold=CONFIG["density_threshold"]
)

# 🔥 ARC FIX: limit to 26 labels
toxcast_labels = toxcast_labels[:26]

toxcast_clean = toxcast_clean[["SMILES"] + toxcast_labels].copy()

print(f"\n📊 Final label counts → Tox21: {len(tox21_labels)} | ToxCast: {len(toxcast_labels)} | SIDER: {len(sider_labels)}")



⚙️  Preprocessing Tox21...
  ✅ Kept 12 labels after NaN filtering
  Final shape: (7823, 13)
  Unique label values: [0. 1.]

⚙️  Preprocessing SIDER...
  ✅ Kept 26 labels after NaN filtering
  Final shape: (1427, 27)
  Unique label values: [0 1]

⚙️  Preprocessing ToxCast...
  ✅ Kept 100 labels after NaN filtering
  Final shape: (8222, 101)
  Unique label values: [0. 1.]
  Density filter: kept 65/100 labels

📊 Final label counts → Tox21: 12 | ToxCast: 26 | SIDER: 26


## 🧱 Section 3: Dataset & Model Architecture

In [7]:
# ============================================================
# CELL 8 — PYTORCH DATASET
# ============================================================
class MolDataset(Dataset):
    """
    Multi-label molecular dataset.
    Returns tokenized SMILES + full label vector.
    Active task labels are selected at training time in the DataLoader.
    """
    def __init__(self, smiles_list, labels_array, tokenizer, max_length=128):
        self.smiles    = smiles_list
        self.labels    = labels_array.astype(np.float32)   # (N, total_labels)
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx]),
        }

print("✅ MolDataset defined")


✅ MolDataset defined


In [8]:
# ============================================================
# COMBINE ALL LABELS
# ============================================================

all_label_names = tox21_labels + sider_labels + toxcast_labels

print("Total labels:", len(all_label_names))

Total labels: 64


In [9]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MLARCModel(nn.Module):
    def __init__(self, model_name="ibm/MoLFormer-XL-both-10pct", num_labels=None):
        super().__init__()

        # 🔥 auto-fix labels
        if num_labels is None:
            num_labels = len(all_label_names)

        self.backbone = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True
        )

        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        # 🔥 FREEZE BACKBONE
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0]
        logits = self.classifier(cls)

        return logits

In [10]:
# ============================================================
# BUILD MODEL FUNCTION
# ============================================================

def build_model(model_name, num_labels):
    
    model = MLARCModel(
        model_name=model_name,
        num_labels=num_labels
    )
    
    return model.to(DEVICE)


print("✅ build_model defined")

✅ build_model defined


In [11]:
model = MLARCModel(num_labels=len(all_label_names))
model.to(DEVICE)

MLARCModel(
  (backbone): MolformerModel(
    (embeddings): MolformerEmbeddings(
      (word_embeddings): Embedding(2362, 768, padding_idx=2)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): MolformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x MolformerLayer(
          (attention): MolformerAttention(
            (self): MolformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (rotary_embeddings): MolformerRotaryEmbedding()
              (feature_map): MolformerFeatureMap(
                (kernel): ReLU()
              )
            )
            (output): MolformerSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (dropo

In [12]:
print(model.classifier.out_features)

64


In [13]:
print("Backbone trainable:",
      any(p.requires_grad for p in model.backbone.parameters()))

print("Classifier trainable:",
      any(p.requires_grad for p in model.classifier.parameters()))

Backbone trainable: False
Classifier trainable: True


## 🔁 Section 4: Sequential Training Pipeline

In [14]:
# ============================================================
# EARLY STOPPING
# ============================================================

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0005):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = model.state_dict()
            return False  # continue training
        else:
            self.counter += 1
            return self.counter >= self.patience  # stop if True

In [15]:
# ============================================================
# TRAIN TASK FUNCTION (ARC FIXED)
# ============================================================

def make_task_dataloader(df, task_label_names, all_label_names, tokenizer,
                          batch_size=32, max_length=128, split="train"):

    task_indices = [all_label_names.index(l) for l in task_label_names]

    smiles = df["SMILES"].tolist()
    labels = df[all_label_names].values.astype(np.float32)

    mask = labels[:, task_indices].sum(axis=1) > 0
    smiles_f = [smiles[i] for i in range(len(smiles)) if mask[i]]
    labels_f = labels[mask]

    dataset = MolDataset(smiles_f, labels_f, tokenizer, max_length)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    return loader, task_indices


def train_task(model, tokenizer, df, task_label_names, all_label_names,
               task_id, epochs=20, batch_size=32):

    print(f"\n🔥 Training Task {task_id+1}")

    loader, task_indices = make_task_dataloader(
        df, task_label_names, all_label_names, tokenizer,
        batch_size=batch_size
    )

    optimizer = AdamW(model.classifier.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    # 🔥 Early stopping
    early_stopper = EarlyStopping(patience=3)

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch in loader:

            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels_all = batch["labels"].to(DEVICE)

            labels_task = labels_all[:, task_indices]

            optimizer.zero_grad()

            logits_all = model(input_ids, attn_mask)
            logits_task = logits_all[:, task_indices]

            loss = criterion(logits_task, labels_task)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.classifier.parameters(), 1.0
            )

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        # 🔥 Early stopping check
        if early_stopper.step(avg_loss, model):
            print(f"⛔ Early stopping triggered at epoch {epoch+1}")
            break

    # 🔥 Restore best model
    if early_stopper.best_state is not None:
        model.load_state_dict(early_stopper.best_state)
        print("✅ Loaded best model weights")

    return model

print("✅ train_task ready")

# TASKS DEFINE (YAHAN LIKHNA HAI)
# ============================================================

tasks = [
    ("tox21", tox21_labels),
    ("sider", sider_labels),
    ("toxcast", toxcast_labels)
]


✅ train_task ready


## 🔥 Section 5: ARC Implementation (Algorithm 1 from Paper)

In [16]:
# ============================================================
# ✅ FINAL FIXED — ARC (OTD + Retention + Correction)
# ============================================================

import torch
import torch.nn.functional as F
from torch.optim import AdamW
import numpy as np


# ------------------------------------------------------------
# OTD (FIXED — correct c and c_hat)
# ------------------------------------------------------------
def otd_detection(probs, past_indices, curr_indices, epsilon, theta):

    if len(past_indices) == 0:
        return 'none', 0.0

    c = probs.max().item()
    c_hat = probs[past_indices].max().item()

    curr_max = probs[curr_indices].max().item() if len(curr_indices) > 0 else 0.0

    # smoother decision
    if c_hat >= curr_max and c > epsilon:
        return 'retention', c

    w = c / (c_hat + 1e-8)
    if c_hat < curr_max and w < theta:
        return 'correction', w

    return 'none', 0.0

# ------------------------------------------------------------
# Adaptive Retention (FIXED — gradient + BCE + entropy)
# ------------------------------------------------------------
def adaptive_retention(model, optimizer, input_ids, attn_mask, past_indices):

    model.train()

    logits = model(input_ids, attn_mask)
    probs = torch.sigmoid(logits).squeeze(0)

    # pseudo labels
    pseudo = (probs > 0.5).float()

    # BCE loss (multi-label)
    L_CE = F.binary_cross_entropy_with_logits(
        logits.squeeze(0)[past_indices],
        pseudo[past_indices]
    )

    # entropy minimization
    p = probs[past_indices].clamp(1e-7, 1 - 1e-7)
    L_EM = -(p * torch.log(p)).mean()

    loss = L_CE + 0.1 * L_EM   # scaled for stability

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), 1.0)
    optimizer.step()

    model.eval()

    with torch.no_grad():
        logits = model(input_ids, attn_mask)
        probs = torch.sigmoid(logits).squeeze(0)

    return probs


# ------------------------------------------------------------
# TSS (minor stability fix)
# ------------------------------------------------------------
def tss_score(logits_np, task_boundaries, temperature):

    scores = []
    t = len(task_boundaries)

    for i, task_idx in enumerate(task_boundaries):

        power = t - i

        cum_idx = [idx for j in range(i + 1) for idx in task_boundaries[j]]

        scaled = logits_np[cum_idx] / (temperature ** power)
        scaled = scaled - np.max(scaled)

        exp_logits = np.exp(scaled)
        softmax_cum = exp_logits / exp_logits.sum()

        n_prev = sum(len(task_boundaries[j]) for j in range(i))
        task_softmax = softmax_cum[n_prev:n_prev + len(task_idx)]

        scores.append(float(task_softmax.max()))

    return scores

# Adaptive Correction
def adaptive_correction(logits_np, task_boundaries, temperature):

    scores = tss_score(logits_np, task_boundaries, temperature)
    best_task = int(np.argmax(scores))

    probs_np = 1 / (1 + np.exp(-logits_np))
    corrected = probs_np.copy()

    # 🔥 smooth suppression instead of zero
    current_task = task_boundaries[-1]
    corrected[current_task] *= 0.5

    return corrected, best_task

# ------------------------------------------------------------
# ARC STEP (FIXED — no graph break)
# ------------------------------------------------------------
def arc_step(model, optimizer, input_ids, attn_mask,
             task_boundaries, current_task_id,
             epsilon=0.6, theta=1.2, temperature=1.5,
             mode='full'):

    model.eval()

    with torch.no_grad():
        logits = model(input_ids, attn_mask)
        probs = torch.sigmoid(logits).squeeze(0)

    probs_np = probs.cpu().numpy()
    logits_np = logits.squeeze(0).cpu().numpy()

    if current_task_id == 0:
        return probs_np, 'none'

    past_indices = [idx for t in range(current_task_id)
                    for idx in task_boundaries[t]]
    curr_indices = task_boundaries[current_task_id]

    case, _ = otd_detection(probs, past_indices, curr_indices, epsilon, theta)

    if mode == 'none':
        return probs_np, 'none'

    # 🔥 RETENTION
    if case == 'retention' and mode in ('full', 'retention_only'):
        updated_probs = adaptive_retention(
            model, optimizer, input_ids, attn_mask, past_indices
        )
        return updated_probs.detach().cpu().numpy(), 'retention'

    # 🔥 CORRECTION
    if case == 'correction' and mode in ('full', 'correction_only'):
        corrected, _ = adaptive_correction(
            logits_np, task_boundaries[:current_task_id+1], temperature
        )
        return corrected, 'correction'

    return probs_np, 'none'

print("✅ FINAL ARC FIXED — READY FOR TRAINING")

✅ FINAL ARC FIXED — READY FOR TRAINING


In [17]:
optimizer = AdamW(model.classifier.parameters(), lr=1e-4)

## 🧪 Section 6: Evaluation Metrics

In [18]:
# ============================================================
# FINAL CELL — WITH & WITHOUT ARC EVALUATION
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np
import torch

# ============================================================
# 1. PER-TASK AUROC (WITH / WITHOUT ARC)
# ============================================================

def evaluate_task_auroc(model, df, task_label_names, all_label_names, tokenizer,
                        task_boundaries=None, current_task_id=None,
                        arc_mode=None):

    task_indices = [all_label_names.index(l) for l in task_label_names]

    smiles = df["SMILES"].tolist()
    labels = df[all_label_names].values.astype(np.float32)

    all_preds, all_true = [], []

    model.eval()

    for smi, label in zip(smiles, labels):

        enc = tokenizer(
            smi,
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attn_mask = enc["attention_mask"].to(DEVICE)

        # 🔥 WITH ARC
        if arc_mode is not None:

            probs_np, _ = arc_step(
                model,
                optimizer,
                input_ids,
                attn_mask,
                task_boundaries,
                current_task_id,
                epsilon=CONFIG["epsilon"],
                theta=CONFIG["theta"],
                temperature=CONFIG["tss_temp"],
                mode=arc_mode
            )

        # 🔹 WITHOUT ARC
        else:
            with torch.no_grad():
                logits = model(input_ids, attn_mask)
                probs_np = torch.sigmoid(logits).squeeze(0).cpu().numpy()

        all_preds.append(probs_np[task_indices])
        all_true.append(label[task_indices])

    all_preds = np.array(all_preds)
    all_true = np.array(all_true)

    # AUROC
    aucs = []
    for i in range(all_true.shape[1]):
        if len(np.unique(all_true[:, i])) < 2:
            continue
        try:
            auc = roc_auc_score(all_true[:, i], all_preds[:, i])
            aucs.append(auc)
        except:
            continue

    return float(np.mean(aucs)) if aucs else 0.0


# ============================================================
# 2. AB + FORGETTING
# ============================================================

def compute_metrics(R_matrix):

    T = len(R_matrix)

    AB = np.mean([
        R_matrix[T-1][i]
        for i in range(T)
        if R_matrix[T-1][i] is not None
    ])

    if T > 1:
        F_vals = []
        for i in range(T - 1):
            if R_matrix[i][i] is not None and R_matrix[T-1][i] is not None:
                F_vals.append(R_matrix[i][i] - R_matrix[T-1][i])
        F = float(np.mean(F_vals)) if F_vals else 0.0
    else:
        F = 0.0

    return AB, F


# ============================================================
# 3. FINAL EVALUATION (WITH vs WITHOUT ARC)
# ============================================================

def evaluate_final_with_arc(model, test_df, all_label_names, task_splits, tokenizer):

    T = len(task_splits)

    # Build task boundaries
    task_boundaries = [
        [all_label_names.index(l) for l in task_splits[t]]
        for t in range(T)
    ]

    # Matrices
    R_no_arc  = [[None]*T for _ in range(T)]
    R_arc     = [[None]*T for _ in range(T)]

    # 🔹 WITHOUT ARC
    print("\n==============================")
    print("🚫 WITHOUT ARC")
    print("==============================")

    for t in range(T):
        for i in range(t + 1):

            auroc = evaluate_task_auroc(
                model,
                test_df,
                task_splits[i],
                all_label_names,
                tokenizer,
                use_arc=False
            )

            R_no_arc[t][i] = auroc
            print(f"Task {i+1} AUROC: {auroc:.4f}")

    AB_no, F_no = compute_metrics(R_no_arc)

    # 🔥 WITH ARC
    print("\n==============================")
    print("🔥 WITH ARC")
    print("==============================")

    for t in range(T):
        for i in range(t + 1):

            auroc = evaluate_task_auroc(
                model,
                test_df,
                task_splits[i],
                all_label_names,
                tokenizer,
                task_boundaries=task_boundaries,
                current_task_id=T-1,
                use_arc=True
            )

            R_arc[t][i] = auroc
            print(f"Task {i+1} AUROC: {auroc:.4f}")

    AB_arc, F_arc = compute_metrics(R_arc)

    # ========================================================
    # FINAL COMPARISON
    # ========================================================

    print("\n🔥 FINAL COMPARISON")
    print(f"NO ARC  → AB: {AB_no:.4f}, F: {F_no:.4f}")
    print(f"ARC     → AB: {AB_arc:.4f}, F: {F_arc:.4f}")

    return {
        "no_arc": {"R": R_no_arc, "AB": AB_no, "F": F_no},
        "arc": {"R": R_arc, "AB": AB_arc, "F": F_arc}
    }


print("✅ WITH & WITHOUT ARC EVALUATION READY")

✅ WITH & WITHOUT ARC EVALUATION READY


## 🚀 Section 7: Full Experiment Runner

In [19]:
# ============================================================
# FINAL — FULL EXPERIMENT RUNNER (CLEAN + FIXED)
# ============================================================

def run_experiment(dataset_name, df, all_label_names, task_splits,
                   epochs=None, arc_modes=(None, "full", "retention_only", "correction_only")):

    if epochs is None:
        epochs = CONFIG["epochs"]

    print(f"\n{'='*60}")
    print(f"🔬 EXPERIMENT: {dataset_name} | {len(all_label_names)} labels | {len(task_splits)} tasks")
    print(f"{'='*60}")

    T = len(task_splits)
    n_labels = len(all_label_names)

    # 🔹 Task boundaries
    task_boundaries = [
        [all_label_names.index(l) for l in task_splits[t]]
        for t in range(T)
    ]

    # 🔹 Model + tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["molformer_name"],
        trust_remote_code=True
    )

    model = build_model(CONFIG["molformer_name"], n_labels)

    # 🔹 Train/test split
    train_df, test_df = train_test_split(
        df, test_size=0.15, random_state=CONFIG["seed"]
    )

    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # ========================================================
    # 1. TRAINING (NO ARC)
    # ========================================================

    R_matrix_base = [[None]*T for _ in range(T)]

    for t in range(T):

        print(f"\n🔥 Training Task {t+1}")

        model = train_task(
            model,
            tokenizer,
            train_df,
            task_splits[t],
            all_label_names,
            task_id=t,
            epochs=epochs,
            batch_size=CONFIG["batch_size"]
        )

        # 🔹 Evaluate WITHOUT ARC
        for i in range(t + 1):

            auroc_i = evaluate_task_auroc(
                model,
                test_df,
                task_splits[i],
                all_label_names,
                tokenizer,
                arc_mode=None   # ✅ NO ARC
            )

            R_matrix_base[t][i] = auroc_i
            print(f"  [After Task {t+1}] Task {i+1} AUROC (no ARC): {auroc_i:.4f}")

    # ========================================================
    # 2. ARC EVALUATION (ONLY TEST TIME)
    # ========================================================

    results = {}

    for mode in arc_modes:

        print(f"\n📊 Evaluating ARC mode: [{mode}]")

        R_matrix = copy.deepcopy(R_matrix_base)
        task_aurocs = []

        for i in range(T):

            auroc_i = evaluate_task_auroc(
                model,
                test_df,
                task_splits[i],
                all_label_names,
                tokenizer,
                task_boundaries=task_boundaries,
                current_task_id=T - 1,
                arc_mode=mode   # ✅ ARC applied here
            )

            R_matrix[T-1][i] = auroc_i
            task_aurocs.append(auroc_i)

            print(f"  Task {i+1} AUROC: {auroc_i:.4f}")

        AB, F = compute_metrics(R_matrix)

        print(f"  ✅ AB (Avg Accuracy): {AB:.4f} | F (Forgetting): {F:.4f}")

        results[str(mode)] = {
            "R_matrix": R_matrix,
            "task_aurocs": task_aurocs,
            "AB": AB,
            "F": F,
        }

    return results, model, tokenizer,  task_boundaries, test_df


print("✅ FINAL run_experiment READY")

✅ FINAL run_experiment READY


## ▶️  Section 8: Run All Experiments
> ⚠️ This cell requires GPU and HuggingFace access. Set `epochs` in CONFIG (default=5).

In [21]:
results, model, tokenizer, task_boundaries, test_df = run_experiment(
    dataset_name="ToxCast",
    df=toxcast_clean,
    all_label_names=toxcast_labels,
    task_splits=[
        toxcast_labels[:8],
        toxcast_labels[8:17],
        toxcast_labels[17:]
    ],
    epochs=CONFIG["epochs"]
)


🔬 EXPERIMENT: ToxCast | 26 labels | 3 tasks

🔥 Training Task 1

🔥 Training Task 1
Epoch 1 Loss: 0.6086
Epoch 2 Loss: 0.5256
Epoch 3 Loss: 0.5090
Epoch 4 Loss: 0.4972
Epoch 5 Loss: 0.4902
Epoch 6 Loss: 0.4840
Epoch 7 Loss: 0.4808
Epoch 8 Loss: 0.4767
Epoch 9 Loss: 0.4737
Epoch 10 Loss: 0.4712
Epoch 11 Loss: 0.4692
Epoch 12 Loss: 0.4655
Epoch 13 Loss: 0.4645
Epoch 14 Loss: 0.4622
Epoch 15 Loss: 0.4606
Epoch 16 Loss: 0.4586
Epoch 17 Loss: 0.4582
Epoch 18 Loss: 0.4577
Epoch 19 Loss: 0.4565
Epoch 20 Loss: 0.4555
✅ Loaded best model weights
  [After Task 1] Task 1 AUROC (no ARC): 0.6798

🔥 Training Task 2

🔥 Training Task 2
Epoch 1 Loss: 0.6181
Epoch 2 Loss: 0.5298
Epoch 3 Loss: 0.5125
Epoch 4 Loss: 0.5054
Epoch 5 Loss: 0.4989
Epoch 6 Loss: 0.4891
Epoch 7 Loss: 0.4866
Epoch 8 Loss: 0.4814
Epoch 9 Loss: 0.4837
Epoch 10 Loss: 0.4811
Epoch 11 Loss: 0.4741
Epoch 12 Loss: 0.4751
Epoch 13 Loss: 0.4720
Epoch 14 Loss: 0.4712
Epoch 15 Loss: 0.4682
Epoch 16 Loss: 0.4639
Epoch 17 Loss: 0.4642
Epoch 18

In [ ]:
print(type(model))       # should be MLARCModel
print(type(tokenizer))   # tokenizer

<class '__main__.MLARCModel'>
<class 'transformers_modules.ibm.MolFormer-XL-both-10pct.7b12d946c181a37f6012b9dc3b002275de070314.tokenization_molformer_fast.MolformerTokenizerFast'>


In [23]:
# ============================================================
# RUN — TOX21 EXPERIMENT
# ============================================================

T21_TASKS = [
    tox21_labels[:4],
    tox21_labels[4:8],
    tox21_labels[8:]
]

t21_results, t21_tokenizer, t21_model, t21_bounds, t21_test = run_experiment(
    dataset_name="Tox21",
    df=tox21_clean,
    all_label_names=tox21_labels,
    task_splits=T21_TASKS,
    epochs=CONFIG["epochs"] 
)

print("\n✅ Tox21 experiment complete!")


🔬 EXPERIMENT: Tox21 | 12 labels | 3 tasks

🔥 Training Task 1

🔥 Training Task 1
Epoch 1 Loss: 0.6864
Epoch 2 Loss: 0.5726
Epoch 3 Loss: 0.5259
Epoch 4 Loss: 0.4971
Epoch 5 Loss: 0.4761
Epoch 6 Loss: 0.4606
Epoch 7 Loss: 0.4558
Epoch 8 Loss: 0.4458
Epoch 9 Loss: 0.4401
Epoch 10 Loss: 0.4328
Epoch 11 Loss: 0.4269
Epoch 12 Loss: 0.4241
Epoch 13 Loss: 0.4235
Epoch 14 Loss: 0.4153
Epoch 15 Loss: 0.4181
Epoch 16 Loss: 0.4114
Epoch 17 Loss: 0.4124
Epoch 18 Loss: 0.4095
Epoch 19 Loss: 0.4082
Epoch 20 Loss: 0.4082
✅ Loaded best model weights
  [After Task 1] Task 1 AUROC (no ARC): 0.7362

🔥 Training Task 2

🔥 Training Task 2
Epoch 1 Loss: 0.6237
Epoch 2 Loss: 0.5666
Epoch 3 Loss: 0.5535
Epoch 4 Loss: 0.5471
Epoch 5 Loss: 0.5412
Epoch 6 Loss: 0.5340
Epoch 7 Loss: 0.5340
Epoch 8 Loss: 0.5274
Epoch 9 Loss: 0.5239
Epoch 10 Loss: 0.5228
Epoch 11 Loss: 0.5188
Epoch 12 Loss: 0.5179
Epoch 13 Loss: 0.5128
Epoch 14 Loss: 0.5124
Epoch 15 Loss: 0.5079
Epoch 16 Loss: 0.5100
Epoch 17 Loss: 0.5085
Epoch 18 L

In [24]:
# ============================================================
# RUN — SIDER EXPERIMENT
# ============================================================

SD_TASKS = [
    sider_labels[:9],
    sider_labels[9:18],
    sider_labels[18:]
]


sd_results, sd_tokenizer, sd_model, sd_bounds, sd_test = run_experiment(
    dataset_name="SIDER",
    df=sider_clean,
    all_label_names=sider_labels,
    task_splits=SD_TASKS,
    epochs=CONFIG["epochs"]
)

print("\n✅ SIDER experiment complete!")


🔬 EXPERIMENT: SIDER | 26 labels | 3 tasks

🔥 Training Task 1

🔥 Training Task 1
Epoch 1 Loss: 0.6836
Epoch 2 Loss: 0.5498
Epoch 3 Loss: 0.5080
Epoch 4 Loss: 0.4955
Epoch 5 Loss: 0.4875
Epoch 6 Loss: 0.4820
Epoch 7 Loss: 0.4777
Epoch 8 Loss: 0.4760
Epoch 9 Loss: 0.4735
Epoch 10 Loss: 0.4725
Epoch 11 Loss: 0.4692
Epoch 12 Loss: 0.4667
Epoch 13 Loss: 0.4643
Epoch 14 Loss: 0.4640
Epoch 15 Loss: 0.4639
Epoch 16 Loss: 0.4615
Epoch 17 Loss: 0.4605
Epoch 18 Loss: 0.4585
Epoch 19 Loss: 0.4589
Epoch 20 Loss: 0.4577
✅ Loaded best model weights
  [After Task 1] Task 1 AUROC (no ARC): 0.5805

🔥 Training Task 2

🔥 Training Task 2
Epoch 1 Loss: 0.5911
Epoch 2 Loss: 0.5103
Epoch 3 Loss: 0.4909
Epoch 4 Loss: 0.4814
Epoch 5 Loss: 0.4790
Epoch 6 Loss: 0.4739
Epoch 7 Loss: 0.4708
Epoch 8 Loss: 0.4664
Epoch 9 Loss: 0.4664
Epoch 10 Loss: 0.4647
Epoch 11 Loss: 0.4638
Epoch 12 Loss: 0.4618
Epoch 13 Loss: 0.4589
Epoch 14 Loss: 0.4568
Epoch 15 Loss: 0.4546
Epoch 16 Loss: 0.4538
Epoch 17 Loss: 0.4530
Epoch 18 L

## 🔬 Section 9: Ablation Study Table

In [25]:
# ============================================================
# FINAL — EVALUATION ONLY (NO RETRAINING)
# ============================================================

import numpy as np

# 🔹 1. DEFINE TASK SPLITS (same as training)
task_splits = [
    toxcast_labels[:8],
    toxcast_labels[8:17],
    toxcast_labels[17:]
]

# 🔹 2. CHECK REQUIRED VARIABLES
print("Model:", model is not None)
print("Test DF:", test_df.shape)
print("Tokenizer:", tokenizer is not None)

# 🔹 3. EVALUATION (NO TRAINING)
results = {}

for mode in ["None", "full", "retention_only", "correction_only"]:

    print(f"\n📊 Evaluating mode: {mode}")

    task_aurocs = []

    for i in range(len(task_splits)):

        auc = evaluate_task_auroc(
            model,
            test_df,
            task_splits[i],
            toxcast_labels,          # all_label_names
            tokenizer,
            task_boundaries=task_boundaries,
            current_task_id=len(task_splits) - 1,
            arc_mode=mode
        )

        task_aurocs.append(auc)

        print(f"Task {i+1} AUROC: {auc:.4f}")

    AB = float(np.mean(task_aurocs))

    results[mode] = {
        "task_aurocs": task_aurocs,
        "AB": AB
    }

    print(f"✅ AB: {AB:.4f}")

print("\n🔥 Evaluation Complete (No Retraining)")

Model: True
Test DF: (1234, 27)
Tokenizer: True

📊 Evaluating mode: None
Task 1 AUROC: 0.6768
Task 2 AUROC: 0.6776
Task 3 AUROC: 0.6992
✅ AB: 0.6846

📊 Evaluating mode: full
Task 1 AUROC: 0.6771
Task 2 AUROC: 0.6791
Task 3 AUROC: 0.6961
✅ AB: 0.6841

📊 Evaluating mode: retention_only
Task 1 AUROC: 0.6807
Task 2 AUROC: 0.6863
Task 3 AUROC: 0.7062
✅ AB: 0.6910

📊 Evaluating mode: correction_only
Task 1 AUROC: 0.6744
Task 2 AUROC: 0.6749
Task 3 AUROC: 0.6946
✅ AB: 0.6813

🔥 Evaluation Complete (No Retraining)


In [41]:
tox21_labels = [col for col in tox21_test_df.columns if col != "SMILES"]

print("Total labels:", len(tox21_labels))
print(tox21_labels[:5])

Total labels: 26
['TOX21_ARE_BLA_Agonist_ch1', 'TOX21_ARE_BLA_Agonist_ch2', 'TOX21_ARE_BLA_agonist_ratio', 'TOX21_AR_BLA_Agonist_ch2', 'TOX21_AR_BLA_Agonist_ratio']


In [44]:
task_boundaries = [
    list(range(0, 8)),
    list(range(8, 17)),
    list(range(17, len(tox21_labels)))
]

In [ ]:
for mode in ["None", "full", "retention_only", "correction_only"]:

    print(f"\n📊 Evaluating mode: {mode}")

    task_aurocs = []

    for i in range(len(task_splits)):

        auc = evaluate_task_auroc(
            model,
            tox21_test_df,
            task_splits[i],
            tox21_labels,   # ✅ now correct
            tokenizer,
            task_boundaries=task_boundaries,
            current_task_id=2,
            arc_mode=mode
        )

        task_aurocs.append(auc)
        print(f"Task {i+1} AUROC: {auc:.4f}")

    print("AB:", sum(task_aurocs)/len(task_aurocs))


📊 Evaluating mode: None
Task 1 AUROC: 0.6839
Task 2 AUROC: 0.6842
Task 3 AUROC: 0.7050
AB: 0.6910157589851295

📊 Evaluating mode: full
Task 1 AUROC: 0.6804
Task 2 AUROC: 0.6735
Task 3 AUROC: 0.6904
AB: 0.6814220741288018

📊 Evaluating mode: retention_only


In [ ]:
task_boundaries = [
    list(range(0, 8)),
    list(range(8, 17)),
    list(range(17, len(sider_labels)))
]


In [ ]:
results_sider = {}

for mode in ["None", "full", "retention_only", "correction_only"]:

    print(f"\n📊 Evaluating mode: {mode}")

    task_aurocs = []

    for i in range(len(task_splits)):

        auc = evaluate_task_auroc(
            model,
            sider_test_df,
            task_splits[i],
            sider_labels,
            tokenizer,
            task_boundaries=task_boundaries,
            current_task_id=2,
            arc_mode=mode
        )

        task_aurocs.append(auc)
        print(f"Task {i+1} AUROC: {auc:.4f}")

    AB = float(np.mean(task_aurocs))

    results_sider[mode] = {
        "task_aurocs": task_aurocs,
        "AB": AB
    }

    print(f"✅ AB: {AB:.4f}")

print("\n🔥 SIDER Evaluation Complete")

In [27]:

# ============================================================
# CELL — ABLATION TABLE (FIXED)
# ============================================================

def print_ablation_table(results_dict):

    rows = []

    mode_labels = {
        "None":             "No ARC (Baseline)",
        "retention_only":   "Retention Only",
        "correction_only":  "Correction Only",
        "full":             "Full ARC",
    }

    print("\n" + "="*85)
    print(f"{'METHOD':<25} {'DATASET':<12} {'AUROC':<10} {'AB':<10} {'FORGETTING':<10}")
    print("="*85)

    for dataset, res in results_dict.items():

        for mode in ["None", "retention_only", "correction_only", "full"]:

            if mode not in res:
                continue

            r = res[mode]

            avg_auroc = float(np.nanmean(r["task_aurocs"]))

            print(f"{mode_labels[mode]:<25} {dataset:<12} {avg_auroc:<10.4f} {r['AB']:<10.4f} {r['F']:<10.4f}")

            rows.append({
                "Method": mode_labels[mode],
                "Dataset": dataset,
                "AUROC": round(avg_auroc, 4),
                "AB": round(r["AB"], 4),
                "Forgetting": round(r["F"], 4),
            })

        print("-"*85)

    return pd.DataFrame(rows)


# 🔥 RUN THIS AFTER ALL EXPERIMENTS
ablation_df = print_ablation_table(all_results)

print("\n📋 Ablation DataFrame:")
print(ablation_df.to_string(index=False))


NameError: name 'all_results' is not defined

## 📈 Section 10: Visualizations

In [ ]:
# ============================================================
# FINAL — 4 CLEAN PLOTS (ALL DATASETS)
# ============================================================

def plot_final_results(all_results):

    datasets = list(all_results.keys())

    # -----------------------------
    # PLOT 1 — PER TASK AUROC
    # -----------------------------
    plt.figure(figsize=(12, 6))

    for d in datasets:
        if "None" in all_results[d]:
            vals = all_results[d]["None"]["task_aurocs"]
            plt.plot(range(1, len(vals)+1), vals, marker='o', label=f"{d} (No ARC)")

        if "full" in all_results[d]:
            vals = all_results[d]["full"]["task_aurocs"]
            plt.plot(range(1, len(vals)+1), vals, linestyle='--', marker='x', label=f"{d} (ARC)")

    plt.title("Per-Task AUROC (All Datasets)")
    plt.xlabel("Task")
    plt.ylabel("AUROC")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.ylim(0, 1)
    plt.show()

    # -----------------------------
    # PLOT 2 — ARC vs NO ARC (AB)
    # -----------------------------
    plt.figure(figsize=(10, 5))

    x = np.arange(len(datasets))
    width = 0.3

    no_arc = [all_results[d]["None"]["AB"] for d in datasets]
    arc    = [all_results[d]["full"]["AB"] for d in datasets]

    plt.bar(x, no_arc, width, label="No ARC")
    plt.bar(x + width, arc, width, label="Full ARC")

    plt.xticks(x + width/2, datasets)
    plt.ylabel("Average Accuracy (AB)")
    plt.title("ARC vs No ARC (Average Accuracy)")
    plt.legend()
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.show()

    # -----------------------------
    # PLOT 3 — FORGETTING
    # -----------------------------
    plt.figure(figsize=(10, 5))

    no_arc_F = [all_results[d]["None"]["F"] for d in datasets]
    arc_F    = [all_results[d]["full"]["F"] for d in datasets]

    plt.bar(x, no_arc_F, width, label="No ARC")
    plt.bar(x + width, arc_F, width, label="Full ARC")

    plt.xticks(x + width/2, datasets)
    plt.ylabel("Forgetting ↓")
    plt.title("Forgetting (ARC vs No ARC)")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()

    # -----------------------------
    # PLOT 4 — RETENTION vs CORRECTION
    # -----------------------------
    plt.figure(figsize=(10, 5))

    retention = [all_results[d]["retention_only"]["AB"] for d in datasets]
    correction = [all_results[d]["correction_only"]["AB"] for d in datasets]

    x2 = np.arange(len(datasets))
    w = 0.25

    plt.bar(x2, retention, w, label="Retention")
    plt.bar(x2 + w, correction, w, label="Correction")
    

    plt.xticks(x2 + w, datasets)
    plt.ylabel("Average Accuracy (AB)")
    plt.title("Retention vs Correction")
    plt.legend()
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)


    # value labels (optional but professional)
    for i in range(len(datasets)):
        plt.text(i, retention[i] + 0.005, f"{retention[i]:.3f}", ha='center')
        plt.text(i + w, correction[i] + 0.005, f"{correction[i]:.3f}", ha='center')

    plt.show()



# 🔥 RUN
plot_final_results(all_results)

## 💾 Section 11: Save Results

In [ ]:
# ============================================================
# CELL 17 — SAVE RESULTS & MODEL CHECKPOINTS
# ============================================================

import json, os

# Save ablation table as CSV
ablation_df.to_csv("arc_ablation_results.csv", index=False)
print("✅ Saved: arc_ablation_results.csv")

# Save numeric results as JSON
def convert(o):
    if isinstance(o, (np.float32, np.float64)):  return float(o)
    if isinstance(o, (np.int32, np.int64)):       return int(o)
    raise TypeError

json_res = {}
for ds, res in all_results.items():
    json_res[ds] = {}
    for mode, v in res.items():
        json_res[ds][mode] = {
            "task_aurocs": [float(x) for x in v["task_aurocs"]],
            "AB": float(v["AB"]),
            "F":  float(v["F"]),
        }

with open("arc_results.json", "w") as f:
    json.dump(json_res, f, indent=2)
print("✅ Saved: arc_results.json")

# Save ToxCast model (main experiment)
torch.save(tc_model.state_dict(), "arc_toxcast_model.pt")
print("✅ Saved: arc_toxcast_model.pt")

# Final summary
print("\n" + "="*60)
print("📋 FINAL SUMMARY (ToxCast — Main Experiment)")
print("="*60)
for mode in ["none", "retention_only", "correction_only", "full"]:
    if mode in all_results["ToxCast"]:
        r = all_results["ToxCast"][mode]
        name = {"none":"Baseline","retention_only":"Retention","correction_only":"Correction","full":"Full ARC"}[mode]
        print(f"  {name:<18} | AB: {r['AB']:.4f} | Forgetting: {r['F']:.4f}")
